In [1]:
from fastapi import FastAPI, UploadFile, File, Form, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from dotenv import load_dotenv
from contextlib import asynccontextmanager

import os
# import httpx
import uvicorn
import threading


# .py 수정 필요!
from service.arrival_service import getRealtimeArrival
from service.position_service import getRealtimePosition
from service.congestion_service import getStationCongestion
from service.database import getSubwayMasterList, initCongestionTableFromCsv

In [2]:
load_dotenv(dotenv_path="../dataset/config/.env")

# startup
@asynccontextmanager
async def lifespan(app: FastAPI):
    # startup
    csvFilePath = "../dataset/서울교통공사_지하철혼잡도정보_20251130.csv"
    
    if os.path.exists(csvFilePath):
        initCongestionTableFromCsv(csvFilePath)
    else:
        print("ℹ️ 적재할 CSV 파일이 지정된 경로에 없습니다.")

    yield

    # shutdown (필요하면 여기에)
    print("서버 종료")

app = FastAPI(lifespan=lifespan, title="BE ETL Server")

app.add_middleware(
    CORSMiddleware,
    # allow_origins=["*"],
    allow_origins=["http://127.0.0.1:5500", "http://localhost:5500"], # 실제 주소를 명시
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

In [3]:
# 노선명 색상 불러오기

@app.get("/subway/lineInfo")
async def get_subway_info():
    lineInfo = await getSubwayMasterList()
    
    return {
        "lineInfo": lineInfo,
    }

In [4]:
# 노선의 지하철 실시간 위치+ 특정 역 도착 정보 호출

@app.get("/subway/realtime")
async def get_subway_info(subway_nm: str, statn_nm: str):
    """
    입력받은 호선의 전체 열차 위치와 
    특정 역의 도착 정보를 한 번에 반환
    """
    # 1. 해당 호선의 전체 열차 위치 정보 가져오기
    positions = await getRealtimePosition(subway_nm)
    
    # 2. 해당 역의 실시간 도착 정보 가져오기
    arrivals = await getRealtimeArrival(statn_nm)
    
    return {
        "subwayName": subway_nm,
        "stationName": statn_nm,
        "trainPositions": positions,
        "arrivalInfo": arrivals,
    }

In [ ]:
# 특정 역 혼잡도 정보 호출
@app.get("/subway/congestion")
def get_congestion(subway_nm: str, statn_nm: str, day_type: str):
    """
    노선과 역명을 받아 해당 역의 혼잡도 통계 반환
    day_type: 평일/ 토요일/ 일요일
    """
    
    congestion_data = getStationCongestion(subway_nm, statn_nm, day_type)
    
    return congestion_data

In [ ]:
def run_server():
    # .env에서 포트를 가져오거나 기본값 8001 사용
    port = int(os.getenv("MAIN_PORT", 3000))

    # 노트북 충돌 방지를 위해 uvicorn.run 사용
    uvicorn.run(app, host="0.0.0.0", port=port, log_level="info")

# 이미 서버가 돌고 있는지 확인 (재실행 시 에러 방지용)
# 주피터에서 셀을 여러 번 누를 때를 대비해 thread가 살아있는지 체크하면 좋습니다.
ocr_thread = threading.Thread(target=run_server, daemon=True)
ocr_thread.start()

print(f"🚀 메인 게이트웨이 서버가 백그라운드에서 시작되었습니다!")
print(f"문서 주소: http://localhost:{os.getenv('MAIN_PORT', 3000)}/docs")

🚀 메인 게이트웨이 서버가 백그라운드에서 시작되었습니다!
문서 주소: http://localhost:3000/docs


INFO:     Started server process [12036]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:3000 (Press CTRL+C to quit)


⚠️ 20251130 날짜 데이터가 이미 존재합니다. 스킵합니다.
result : ()
()
INFO:     127.0.0.1:53445 - "GET /subway/congestion?subway_nm=4%ED%98%B8%EC%84%A0&statn_nm=%EA%B3%BC%EC%B2%9C&day_type=%ED%8F%89%EC%9D%BC HTTP/1.1" 200 OK
result : ()
()
INFO:     127.0.0.1:50318 - "GET /subway/congestion?subway_nm=4%ED%98%B8%EC%84%A0&statn_nm=%EA%B3%BC%EC%B2%9C&day_type=%ED%8F%89%EC%9D%BC HTTP/1.1" 200 OK
result : [{'row_id': 227, 'day_type': '평일', 'line_nm': '4호선', 'station_cd': 424, 'station_nm': '명동', 'direction': '상선', 't0530': 13.0, 't0600': 17.1, 't0630': 15.3, 't0700': 20.9, 't0730': 27.0, 't0800': 39.9, 't0830': 42.1, 't0900': 31.0, 't0930': 34.8, 't1000': 35.4, 't1030': 33.9, 't1100': 31.4, 't1130': 33.1, 't1200': 38.9, 't1230': 43.6, 't1300': 42.8, 't1330': 41.9, 't1400': 45.3, 't1430': 44.9, 't1500': 45.8, 't1530': 53.4, 't1600': 44.0, 't1630': 66.9, 't1700': 77.5, 't1730': 77.7, 't1800': 83.6, 't1830': 67.7, 't1900': 52.0, 't1930': 38.0, 't2000': 35.2, 't2030': 31.9, 't2100': 33.4, 't2130': 28.6, 't2200':